In [ ]:
!pip install pillow
!pip install geopandas
!pip install rasterio --upgrade
!pip install scipy -U
!pip install mkl
!pip install fiona
!pip install C:\Users\flopes1\Downloads\GDAL-3.4.3-cp39-cp39-win_amd64.whl
!pip install ogr
!pip install tqdm

# Convert raster image to a set of tiles

In [ ]:
import geopandas as gpd
import rasterio
import rasterio.features
import numpy as np
from tqdm import tqdm
from osgeo import gdal, ogr

# Abrir a imagem TIFF e o arquivo shapefile
tiff_file = gdal.Open(r"C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Cahn\CHACABUCO\CHACABUCO\LARGO1\10-05-2021\LARGO_1_0924_ALL_QUAC")
shp_file = ogr.Open(r"C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Cahn\CHACABUCO\CHACABUCO\LARGO1\PLOTBOUNDARIES\PLOTBOUNDARIES_PROJECT.shp")
layer = shp_file.GetLayer()

# Criar um novo arquivo TIFF para cada feição do shapefile
for i in tqdm(range(layer.GetFeatureCount())):
    feature = layer.GetFeature(i)
    geometry = feature.GetGeometryRef()
    xmin, xmax, ymin, ymax = geometry.GetEnvelope()

    # Definir as opções do recorte
    options = gdal.WarpOptions(
        outputBounds=(xmin, ymin, xmax, ymax),
        format='GTiff',
        cutlineDSName=r"C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Cahn\CHACABUCO\CHACABUCO\LARGO1\PLOTBOUNDARIES\PLOTBOUNDARIES_PROJECT.shp",
        cropToCutline=True,
        cutlineWhere=f"OBJECTID='{feature.GetField('OBJECTID')}'"
    )

    # Salvar a imagem recortada em um novo arquivo TIFF
    gdal.Warp(f"imagem_segmentada_{feature.GetField('OBJECTID')}_0924_LARGO1_{feature.GetField('HEALTH_STA')}.tif", tiff_file, options=options)


# Use the tiles as input for training a GAN

### Define o conjunto de dados a ser utilizado

In [ ]:
!pip install tiff
!pip install torch
!pip install torchvision
!pip install pytorch-lightning


In [ ]:
!pip install argparse

In [ ]:
!pip install libtiff

In [ ]:
!pip install tifffile

In [ ]:
!pip install opencv-python

In [ ]:
!pip install opencv-torchvision-transforms-yuzhiyang --user

In [ ]:
!pip install tensorflow

In [ ]:
!pip install tensorflow

In [ ]:
# Import the required libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

# Define the generator model using Keras
generator = keras.Sequential(
    [
        keras.Input(shape=(100,)),
        layers.Dense(256),
        layers.LeakyReLU(alpha=0.2),
        layers.BatchNormalization(momentum=0.8),
        layers.Dense(512),
        layers.LeakyReLU(alpha=0.2),
        layers.BatchNormalization(momentum=0.8),
        layers.Dense(1024),
        layers.LeakyReLU(alpha=0.2),
        layers.BatchNormalization(momentum=0.8),
        layers.Dense(300 * 300 * 3, activation="tanh"),
        layers.Reshape((300, 300, 3)),
    ],
    name="generator",
)

# Define the discriminator model using Keras
discriminator = keras.Sequential(
    [
        keras.Input(shape=(300, 300, 3)),
        layers.Flatten(),
        layers.Dense(512),
        layers.LeakyReLU(alpha=0.2),
        layers.Dense(256),
        layers.LeakyReLU(alpha=0.2),
        layers.Dense(1, activation="sigmoid"),
    ],
    name="discriminator",
)

# Define the GAN model as a combination of generator and discriminator models
discriminator.trainable = False
gan_input = keras.Input(shape=(100,))
gan_output = discriminator(generator(gan_input))
gan = keras.Model(gan_input, gan_output, name="gan")

# Compile the models
generator_optimizer = keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
discriminator_optimizer = keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
discriminator.compile(loss="binary_crossentropy", optimizer=discriminator_optimizer)
gan.compile(loss="binary_crossentropy", optimizer=generator_optimizer)

# Load the .tif images and preprocess them
import os

# Define the root directory where the images are located
root_dir = "./dataset/"

# Recursively find all the .tif files in the root directory and its subdirectories
image_paths = []
for dirpath, _, filenames in os.walk(root_dir):
    for filename in filenames:
        if filename.endswith(".tif"):
            image_path = os.path.join(dirpath, filename)
            image_paths.append(image_path)
            
images = []
for path in image_paths:
    # Load the image using OpenCV
    image = cv2.imread(path)

    # Convert the image to a NumPy array and normalize the pixel values to the range [-1, 1]
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.astype("float32") / 127.5 - 1.0

    # Resize the image to the desired size using OpenCV
    image = cv2.resize(image, (300, 300))

    # Append the preprocessed image to the list of images
    images.append(image)
    
train_images = np.array(images)
train_dataset = tf.data.Dataset.from_tensor_slices(train_images)
train_dataset = train_dataset.shuffle(buffer_size=1024).batch(32)

# Define a function to generate images using the trained GAN model
def generate_images(model, noise, epoch):
    # Generate images from noise using the generator model
    generated_images = model.predict(noise)

    # Rescale pixel values to the range [0, 1]
    generated_images = 0.5 * generated_images + 0.5

    # Save the generated images
    for i in range(generated_images.shape[0]):
        plt.imsave(f"generated_images/{epoch}_{i}.png", generated_images[i])

# Train the GAN model on the preprocessed dataset
epochs = 100
noise_dim = 100
num_examples_to_generate = 16
seed = tf.random.normal([num_examples_to_generate, noise_dim])

for epoch in range(epochs):
    print(f"Epoch {epoch+1}")
    for real_images in train_dataset:
        # Generate random noise
        noise = tf.random.normal([real_images.shape[0], noise_dim])

        # Generate fake images using the generator model
        fake_images = generator.predict(noise)

        # Concatenate real and fake images
        combined_images = tf.concat([real_images, fake_images], axis=0)

        # Create labels for real and fake images
        real_labels = tf.ones((real_images.shape[0], 1))
        fake_labels = tf.zeros((fake_images.shape[0], 1))
        combined_labels = tf.concat([real_labels, fake_labels], axis=0)

        # Train the discriminator model
        discriminator_loss = discriminator.train_on_batch(combined_images, combined_labels)

        # Train the generator model
        noise = tf.random.normal([real_images.shape[0], noise_dim])
        generator_loss = gan.train_on_batch(noise, real_labels)

    # Generate images using the trained GAN model
    if epoch % 10 == 0:
        generate_images(generator, seed, epoch)

# Save the generator model
generator.save("generator_model.h5")

In [ ]:
import os
from os import makedirs

# PyTorch packages
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils

from PIL import Image

# Typing
from torch.utils.data.dataloader import DataLoader
from torch.optim.optimizer import Optimizer
from torch.utils.data import Dataset
from torch.functional import Tensor
from typing import Dict, Tuple, List

from cvtorchvision import cvtransforms

# PyTorch Lightning
import pytorch_lightning as pl

# Output Folder for the files
path = './output_lightning'

# create output folder if doesn't exist
makedirs(path, exist_ok=True)

# shape of the image (gray)
img_shape = (1, 462, 312)


# Generator Model
class Generator(nn.Module):
    def __init__(self, latent_dim=100):
        super(Generator, self).__init__()

        self.fc = nn.Sequential(
            nn.Linear(latent_dim, 64*29*20),
            nn.LeakyReLU(0.2, inplace=True),
        )

        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            nn.ConvTranspose2d(32, 16, 4, 2, 1, bias=False),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.2, inplace=True),

            nn.ConvTranspose2d(16, 5, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.fc(x)
        x = x.view(x.size(0), 64, 29, 20)
        x = self.deconv(x)
        return x

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.conv1 = nn.Conv2d(5, 16, 3, stride=2, padding=1)  # 233x156
        self.conv2 = nn.Conv2d(16, 32, 3, stride=2, padding=1)  # 117x78
        self.conv3 = nn.Conv2d(32, 64, 3, stride=2, padding=1)  # 59x40
        self.conv4 = nn.Conv2d(64, 1, 3, stride=1, padding=1)  # 59x40
        
    def forward(self, x):
        x = F.leaky_relu(self.conv1(x), 0.2)
        x = F.leaky_relu(self.conv2(x), 0.2)
        x = F.leaky_relu(self.conv3(x), 0.2)
        x = torch.sigmoid(self.conv4(x))
        x = x.view(x.size(0), -1)
        return x


# Lightning Module
class GAN(pl.LightningModule):
    def __init__(self, hparams) -> None:
        super(GAN, self).__init__()

        self.hparams.lr = hparams.lr
        self.hparams.batch_size = hparams.batch_size
        self.generator = Generator()
        self.discriminator = Discriminator()

    def forward(self, x) -> Tensor:
        return self.discriminator(x)

    def loss_function(self, y_hat, y) -> Tensor:
        return nn.BCELoss()(y_hat, y)

    def configure_optimizers(self) -> Tuple[List[Optimizer], List]:
        optimizer_G = torch.optim.Adam(self.generator.parameters(), lr=self.hparams.lr, betas=(0.4, 0.999))
        optimizer_D = torch.optim.Adam(self.discriminator.parameters(), lr=self.hparams.lr, betas=(0.4, 0.999))

        return [optimizer_G, optimizer_D], []

    def prepare_data(self) -> Dataset:
        transform = cvtransforms.Compose([cvtransforms.ToTensor(),
                                        cvtransforms.Normalize([0.5], [0.5])])
        transform = cvtransforms.Compose([
            cvtransforms.Resize((466, 312)),
            cvtransforms.ToTensor()
        ])

        # Instancia o dataset com o diretório raiz "C:/my_dataset"
        train_data = MyDataset(r'C:/Users/flopes1/OneDrive - Saint Louis University/Desktop/Repos/plant-disease-prediction/',
                               transform=transform)
        print(train_data)
        """
            train_data = datasets.MNIST('./data',
                                    train=True,
                                    download=False,
                                    transform=transform)
        """
        return train_data

    def train_dataloader(self) -> DataLoader:
        train_data = self.prepare_data()
        train_loader = DataLoader(train_data,
                                  batch_size=self.hparams.batch_size,
                                  shuffle=True)
        return train_loader

    def training_step(self, batch, batch_idx, optimizer_idx) -> Dict:
        real_images, _ = batch
        valid = torch.ones(real_images.size(0), 1)
        fake = torch.zeros(real_images.size(0), 1)
        criterion = self.loss_function

        if optimizer_idx == 0:
            gen_input = torch.randn(real_images.shape[0], 100)
            self.gen_images = self.generator(gen_input)

            g_loss = criterion(
                self(self.gen_images), valid)

            tqdm_dict = {'g_loss': g_loss}
            output = {
                'loss': g_loss,
                'progress_bar': tqdm_dict,
                'log': tqdm_dict,
                'g_loss': g_loss
            }
            return output

        if optimizer_idx == 1:
            real_loss = criterion(
                self(real_images), valid)
            fake_loss = criterion(
                self(self.gen_images.detach()), fake)
            d_loss = (real_loss + fake_loss) / 2.0

            tqdm_dict = {'d_loss': d_loss}
            output = {
                'loss': d_loss,
                'progress_bar': tqdm_dict,
                'log': tqdm_dict,
                'd_loss': d_loss
            }
            return output

    def on_train_epoch_end(self) -> None:
        utils.save_image(self.gen_images.data[:25],
                         path + '/%d.tif' % self.current_epoch,
                         nrow=5,
                         padding=0,
                         normalize=True)


from argparse import ArgumentParser

# Hyperparameters
parser = ArgumentParser(description='GAN Wheat dataset')
parser = pl.Trainer.add_argparse_args(parser)
parser.add_argument('--batch_size', type=int, default=32)
parser.add_argument('--lr', type=float, default=2e-4)
parser.add_argument('-f', required=False)

args = parser.parse_args()

# Model Initialization
gan = GAN(hparams=args)

delattr(args, 'f')
# Model Training
trainer = pl.Trainer.from_argparse_args(args,
                                        max_epochs=20,
                                        fast_dev_run=True)

trainer.fit(gan)

In [ ]:
from torchvision import transforms

# Cria uma transformação que dimensiona as imagens para 466x312 pixels e as converte em tensores
transform = transforms.Compose([
    transforms.Resize((466, 312)),
    transforms.ToTensor()
])

# Instancia o dataset com o diretório raiz "C:/my_dataset"
dataset = MyDataset(root_dir='C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Repos\plant-disease-prediction', transform=transform)

# Usa o DataLoader para carregar os dados em lotes de tamanho 32, embaralhando os dados para evitar overfitting
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)